# PV Vision AI - Colab Resume (6/10 -> 10/10)

Önce **Çalışma zamanı > Çalışma zamanı türünü değiştir > GPU** seçin. Hücreleri sırayla çalıştırın. Eğitim test setini kullanmaz ve her tamamlanan epoch'u Google Drive'a kaydeder.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA kullanılabilir:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Yok")
assert torch.cuda.is_available(), "GPU etkin değil. Colab çalışma zamanı ayarından GPU seçin."

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess
import sys

drive.mount("/content/drive")
ROOT = Path("/content/pv_vision_ai")
DRIVE = Path("/content/drive/MyDrive/PV_Vision_AI_Colab")
ARCHIVE = DRIVE / "pv_vision_colab.tar.gz"
assert ARCHIVE.exists(), f"Paket bulunamadı: {ARCHIVE}"

shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(str(ARCHIVE), str(ROOT), format="gztar")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.186", "pyyaml==6.0.2"], check=True)
print("Paket açıldı:", ROOT)

In [ ]:
import json
import os
import shutil
import sys

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from training.prepare_colab import localize_dataset_yaml

RUN_NAME = "pv_vision_yolov8n_mps_v1"
TARGET_EPOCHS = 10
DATASET = ROOT / "data/processed/dataset.yaml"
LOCAL_RUN = ROOT / "outputs/training" / RUN_NAME
DRIVE_PROJECT = DRIVE / "outputs/training"
DRIVE_RUN = DRIVE_PROJECT / RUN_NAME
DRIVE_CHECKPOINT = DRIVE_RUN / "weights/last.pt"

localize_dataset_yaml(DATASET, data_root=ROOT / "data/processed")
if not DRIVE_CHECKPOINT.exists():
    DRIVE_RUN.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(LOCAL_RUN, DRIVE_RUN, dirs_exist_ok=True)
    print("6/10 checkpoint'i Drive'a aktarıldı.")
else:
    print("Drive'daki mevcut checkpoint korunuyor; üzerine eski paket yazılmadı.")

train_images = list((ROOT / "data/processed/images/train").glob("*.jpg"))
val_images = list((ROOT / "data/processed/images/val").glob("*.jpg"))
train_labels = list((ROOT / "data/processed/labels/train").glob("*.txt"))
val_labels = list((ROOT / "data/processed/labels/val").glob("*.txt"))
assert (len(train_images), len(train_labels)) == (3600, 3600)
assert (len(val_images), len(val_labels)) == (900, 900)

checkpoint = torch.load(DRIVE_CHECKPOINT, map_location="cpu", weights_only=False)
completed = int(checkpoint.get("epoch", -1)) + 1
assert 6 <= completed <= TARGET_EPOCHS, f"Beklenmeyen epoch: {completed}"
if completed < TARGET_EPOCHS:
    assert checkpoint.get("optimizer") is not None, "Optimizer bilgisi bulunamadı."
print(f"Hazır: {completed}/{TARGET_EPOCHS} epoch tamamlandı.")
print("Test split'i eğitim paketinde yok.")

In [ ]:
import subprocess
import sys
import torch

checkpoint = torch.load(DRIVE_CHECKPOINT, map_location="cpu", weights_only=False)
completed = int(checkpoint.get("epoch", -1)) + 1
if completed < TARGET_EPOCHS:
    command = [
        sys.executable, "training/train.py",
        "--resume-from", str(DRIVE_CHECKPOINT),
        "--resume-epochs", str(TARGET_EPOCHS),
        "--device", "0",
        "--resume-data", str(DATASET),
        "--resume-project", str(DRIVE_PROJECT),
        "--resume-name", RUN_NAME,
        "--resume-workers", "2",
    ]
    print(f"Eğitim {completed + 1}/10'dan devam ediyor.")
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print("Eğitim zaten 10/10 tamamlanmış.")

In [ ]:
import shutil
import torch

checkpoint = torch.load(DRIVE_CHECKPOINT, map_location="cpu", weights_only=False)
completed = int(checkpoint.get("epoch", -1)) + 1
assert completed == 10, f"Eğitim henüz tamamlanmadı: {completed}/10"
assert (DRIVE_RUN / "weights/best.pt").exists()

drive_model_dir = DRIVE / "models/weights"
drive_model_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE_RUN / "weights/best.pt", drive_model_dir / "best.pt")
print("Tamamlanan epoch: 10/10")
print("best.pt:", drive_model_dir / "best.pt")
print("last.pt:", DRIVE_CHECKPOINT)
print("Eğitim başarıyla tamamlandı; dosyalar Google Drive'da güvende.")

## Sonraki adım
10/10 mesajını gördükten sonra Colab çıktısını kapatmadan önce ekran görüntüsünü alın. Drive'daki `best.pt`, `last.pt` ve `results.csv` dosyaları Mac'teki proje klasörüne geri alınacak; ardından validation kalite kapısı ve resmi test değerlendirmesi çalıştırılacak.